# 🚗 Automated Computer Vision Engine for Car Insurance Claims Assessment
### 4-Phase Multimodal Sequential Pipeline (CNN + SVM/HOG + K-Means + Random Forest)
**Target Platform:** Google Colab (T4 GPU Accelerated) & Local Inference  
**Dataset:** Kaggle Car Damage Severity Dataset (`data3a`)  
**Team Architecture:** 4 Specialized Member Divisions

## 🛠️ Cell 1: Environment Setup & Library Installation
Installs PyTorch, Torchvision, Scikit-Learn, OpenCV, and Scikit-Image.

In [ ]:
!pip install -q torch torchvision scikit-learn opencv-python-headless scikit-image

## 📦 Cell 2: Kaggle Dataset Download & Unpack
Upload your `kaggle.json` or download the dataset directly.

In [ ]:
# Colab Dataset Setup
from google.colab import files
import os
if not os.path.exists('/root/.kaggle/kaggle.json'):
    print('Upload your kaggle.json file:')
    files.upload()
    !mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
    !kaggle datasets download -d l33ts3c/car-damage-severity-dataset -p /content/
    !unzip -q /content/car-damage-severity-dataset.zip -d /content/dataset
print('Dataset ready!')

## 👤 Phase 1: Deep Learning Damage Severity Classifier (Member 1)
**Architecture:** MobileNetV2 Transfer Learning  
**Objective:** Predict damage severity class (`01-minor`, `02-moderate`, `03-severe`) with confidence score.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
import pickle

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Active Device: {device}')

data_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

train_dataset = datasets.ImageFolder('/content/dataset/data3a/training', transform=data_transforms)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

weights = models.MobileNet_V2_Weights.DEFAULT
cnn_model = models.mobilenet_v2(weights=weights)
for param in cnn_model.features.parameters():
    param.requires_grad = False

in_features = cnn_model.classifier[1].in_features
cnn_model.classifier = nn.Sequential(
    nn.Dropout(p=0.2),
    nn.Linear(in_features, 128),
    nn.ReLU(),
    nn.Dropout(p=0.2),
    nn.Linear(128, 3)
)
cnn_model = cnn_model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(filter(lambda p: p.requires_grad, cnn_model.parameters()), lr=0.001)

# Train 5 epochs
for epoch in range(1, 6):
    cnn_model.train()
    running_loss, correct, total = 0.0, 0, 0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        outs = cnn_model(imgs)
        loss = criterion(outs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * imgs.size(0)
        _, preds = torch.max(outs, 1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
    print(f'Epoch {epoch}/5 - Loss: {running_loss/total:.4f} | Acc: {correct/total*100:.2f}%')

torch.save(cnn_model.state_dict(), '/content/cnn_model.pth')
print('Saved /content/cnn_model.pth')

## 👤 Phase 2: Classical SVM + HOG Structural Frame Deformation Detector (Member 2)
**Algorithm:** RBF-Kernel Support Vector Machine + Histogram of Oriented Gradients (HOG)
**Objective:** Extract edge gradient vectors to classify structural deformation (`1: Deformed`, `0: Intact`).

In [ ]:
import cv2
from skimage.feature import hog
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score
import numpy as np

def extract_hog(img_path):
    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    resized = cv2.resize(img, (128, 128))
    return hog(resized, orientations=9, pixels_per_cell=(8, 8), cells_per_block=(2, 2))

X_svm, y_svm = [], []
for c_idx, folder in [(0, '01-minor'), (2, '03-severe')]:
    p = f'/content/dataset/data3a/training/{folder}'
    for f in os.listdir(p)[:150]:
        X_svm.append(extract_hog(os.path.join(p, f)))
        y_svm.append(1 if c_idx == 2 else 0)

svm_clf = SVC(kernel='rbf', probability=True, random_state=42)
svm_clf.fit(X_svm, y_svm)
print('SVM Accuracy:', accuracy_score(y_svm, svm_clf.predict(X_svm)) * 100)
with open('/content/svm_hog_model.pkl', 'wb') as f:
    pickle.dump(svm_clf, f)
print('Saved /content/svm_hog_model.pkl')

## 👤 Phase 3: Unsupervised K-Means Color-Space Surface Area Segmentation (Member 3)
**Algorithm:** K-Means ($k=3$) in HSV Color Space  
**Objective:** Segment damaged paint/metal from bodywork and calculate surface area damage percentage.

In [ ]:
from sklearn.cluster import KMeans

def segment_damage(img_path):
    img = cv2.imread(img_path)
    resized = cv2.resize(img, (160, 160))
    hsv = cv2.cvtColor(resized, cv2.COLOR_BGR2HSV)
    pixels = hsv.reshape((-1, 3)).astype(np.float32)
    kmeans = KMeans(n_clusters=3, n_init=3, random_state=42).fit(pixels)
    counts = np.bincount(kmeans.labels_)
    damage_cluster = np.argsort(counts)[1]
    mask = (kmeans.labels_ == damage_cluster).reshape((160, 160)).astype(np.uint8) * 255
    area_pct = (np.sum(mask > 0) / (160 * 160)) * 100
    return round(area_pct, 2), mask

print('K-Means segmentation module loaded.')

## 👤 Phase 4: Random Forest Claims Cost Estimator & Deployment (Member 4)
**Algorithm:** Random Forest Classifier  
**Objective:** Fuse outputs from Members 1-3 to predict final Claim Cost Tier (`Low`, `Medium`, `High`).

In [ ]:
from sklearn.ensemble import RandomForestClassifier

np.random.seed(42)
X_rf, y_rf = [], []
for _ in range(600):
    c = np.random.choice([0, 1, 2])
    d = 1 if (c == 2 and np.random.rand() > 0.15) else (1 if np.random.rand() > 0.8 else 0)
    a = np.random.uniform(2, 15) if c == 0 else (np.random.uniform(10, 30) if c == 1 else np.random.uniform(25, 60))
    tier = 2 if (d == 1 or c == 2 or a > 30) else (1 if (c == 1 or a > 12) else 0)
    X_rf.append([c, d, a])
    y_rf.append(tier)

rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_rf, y_rf)
with open('/content/random_forest_model.pkl', 'wb') as f:
    pickle.dump(rf, f)
print('Saved /content/random_forest_model.pkl')